# Phase 5: Walk-Forward Backtesting (จำลองการลงทุนจริงรายเดือน 7 ปี)

จำลองการจัดพอร์ตแบบ **Rolling Window 36 เดือน (3 ปี)** และปรับพอร์ตทุกสิ้นเดือน (**Monthly Rebalancing**) ตลอดช่วงปี **2018–2024 (84 เดือน)**
- **Financial Hygiene**: ไม่ใช้ข้อมูลล่วงหน้า (No Look-ahead Bias) น้ำหนักคำนวณจากอดีต แล้วนำไปถือจริงในเดือนถัดไป
- **Transaction Costs**: คิดค่าธรรมเนียมและ Slippage **0.15% (15 bps)** ทุกครั้งที่มีการหมุนเปลี่ยนหุ้น (Turnover)
- เปรียบเทียบ 4 กลยุทธ์ (**EW, IV, Min-Var, Max-Sharpe**) ปะทะ Benchmark (**TDEX.BK** กองทุนดัชนี SET50)

### Step 1: เตรียมข้อมูลและปฏิทิน Rebalance รายเดือน

#### อธิบายการทำงานของโค้ดในเซลล์ถัดไป:
1. **`sys.path.append("..")`**:
   - บอกให้ Python รู้จักโฟลเดอร์หลัก เพื่อให้เรียกใช้โมดูลใน `src/` (`src.data`, `src.portfolio`, `src.risk`) ได้
2. **แยก Benchmark (`TDEX.BK`) ออกจากหุ้นรายตัว**:
   - `prices["TDEX.BK"]`: เก็บไว้เป็นตัวเปรียบเทียบ (Benchmark วัดผล)
   - `prices.drop(columns=["TDEX.BK"])': หุ้น 16 ตัวที่จะนำมาจัดพอร์ตแข่งขัน
3. **`resample("ME").last().index`**:
   - ฟังก์ชันของ Pandas ใช้ดึงวันทำการสุดท้ายของทุกๆ เดือน (Month-Ends) ตลอด 10 ปี (ได้ทั้งหมด 120 เดือน)
4. **แบ่งช่วง In-sample และ Out-of-sample**:
   - `lookback_months = 36`: ต้องใช้ข้อมูลอดีต 36 เดือน (3 ปี) เสมอในการคำนวณน้ำหนักพอร์ต
   - 36 เดือนแรก (2015–2017): ใช้เป็น In-sample ก้อนแรก
   - 84 เดือนที่เหลือ (2018–2024): คือช่วงเวลาที่เราจะทำการจำลองการลงทุนและวัดผลจริง (Out-of-sample Test)

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data import load_price_matrix
from src.portfolio import (
    get_equal_weights,
    get_inverse_vol_weights,
    get_min_variance_weights,
    get_max_sharpe_weights
)
from src.risk import (
    calculate_annualized_return,
    calculate_annualized_volatility,
    calculate_sharpe_ratio,
    calculate_drawdowns,
    calculate_max_drawdown
)

# 1. โหลดข้อมูลราคาหุ้น
prices = load_price_matrix()
bench_prices = prices["TDEX.BK"]
stock_prices = prices.drop(columns=["TDEX.BK"])

# 2. คำนวณ Daily Returns
stock_returns = stock_prices.pct_change().dropna()
bench_returns = bench_prices.pct_change().dropna()

# 3. สร้างตารางสิ้นเดือน (Month-Ends)
month_ends = stock_returns.resample("ME").last().index
lookback_months = 36
test_months = month_ends[lookback_months:]

print(f"จำนวนเดือนทั้งหมด: {len(month_ends)} เดือน (2015-2024)")
print(f"In-sample Lookback: {lookback_months} เดือน (3 ปี)")
print(f"Out-of-sample Test: {len(test_months)} เดือน ({test_months[0].strftime('%Y-%m')} ถึง {test_months[-1].strftime('%Y-%m')})")

### Step 2: กลไก Walk-Forward Loop, การคำนวณ Turnover และค่าธรรมเนียม

#### 1. สูตร Portfolio Turnover (อัตราการหมุนพอร์ต):
- ในแต่ละเดือนเมื่อ Rebalance จากน้ำหนักเดิม $\mathbf{w}_{t-1}$ ไปเป็นน้ำหนักใหม่ $\mathbf{w}_t$ สัดส่วนการซื้อขายคำนวณจาก:
  $$\text{Turnover}_t = \frac{1}{2} \sum_{i=1}^N |w_{i, t} - w_{i, t-1}|$$
- **ทำไมต้องหาร 2?**: เพราะการปรับพอร์ต 100% คือการ **"ขายหุ้นเก่าออก 50% และนำเงินไปซื้อหุ้นใหม่เข้า 50%"** (One-way Turnover)

---

#### 2. สูตร Transaction Cost (ต้นทุนการทำธุรกรรม):
- กำหนดค่าคอมมิชชั่น + ภาษี + Slippage = **0.15% (15 bps)**
- ต้นทุนการเทรดในแต่ละเดือน: $\text{Cost}_t = \text{Turnover}_t \times 0.15\%$
- หัก $\text{Cost}_t$ ออกจากผลตอบแทนของวันแรกในเดือนนั้นทันที ทำให้ผลตอบแทนที่ได้คือ **Net Return** หลังหักค่าใช้จ่ายจริง

---

#### 3. อธิบายการทำงานของ Loop ในเซลล์ถัดไป:
- `for i in range(lookback_months, len(month_ends)):`: ลูปเดินหน้าทีละ 1 เดือน รวม 84 รอบ
- `in_sample = stock_returns.loc[train_start:train_end]`: สไลซ์ข้อมูลย้อนหลัง 36 เดือน ส่งเข้าไปคำนวณหาน้ำหนัก **(รับประกัน No Look-ahead Bias)**
- `out_sample = stock_returns.loc[train_end:test_month].iloc[1:]`: สไลซ์ผลตอบแทนจริงของเดือนถัดไป
- `port_daily_ret = (out_sample @ w)`: นำเมทริกซ์ผลตอบแทนรายวันคูณกับ Vector น้ำหนัก (Matrix Multiplication) จะได้ผลตอบแทนของพอร์ตในแต่ละวัน
- `port_daily_ret.iloc[0] -= cost`: หักค่าธรรมเนียมในวันแรกที่มีการ Rebalance

In [ ]:
# Walk-Forward Backtesting Engine
strategies = ["EW", "IV", "MinVar", "MaxSharpe"]
prev_weights = {s: np.zeros(stock_returns.shape[1]) for s in strategies}

daily_portfolio_returns = {s: [] for s in strategies}
daily_portfolio_returns["Benchmark"] = []
monthly_turnovers = {s: [] for s in strategies}
cost_rate = 0.0015  # 0.15%

print("กำลังรัน Walk-Forward Backtest 84 เดือน...")

for i in range(lookback_months, len(month_ends)):
    train_end = month_ends[i - 1]
    train_start = month_ends[i - lookback_months]
    test_month = month_ends[i]
    
    # 1. In-sample Lookback (36 เดือนย้อนหลัง)
    in_sample = stock_returns.loc[train_start:train_end]
    
    # 2. Out-of-sample ถือในเดือนถัดไป
    out_sample = stock_returns.loc[train_end:test_month].iloc[1:]
    bench_out = bench_returns.loc[train_end:test_month].iloc[1:]
    if len(out_sample) == 0:
        continue
        
    # 3. คำนวณน้ำหนักพอร์ตจาก In-sample เท่านั้น (No Look-ahead)
    weights = {
        "EW": get_equal_weights(in_sample).values,
        "IV": get_inverse_vol_weights(in_sample).values,
        "MinVar": get_min_variance_weights(in_sample).values,
        "MaxSharpe": get_max_sharpe_weights(in_sample).values
    }
    
    # 4. คิด Turnover, หัก Transaction Cost และคำนวณผลตอบแทนรายวัน
    for s in strategies:
        w = weights[s]
        turnover = 0.5 * np.sum(np.abs(w - prev_weights[s]))
        monthly_turnovers[s].append(turnover)
        cost = turnover * cost_rate
        prev_weights[s] = w
        
        # ผลตอบแทนพอร์ต = ผลคูณของผลตอบแทนรายหุ้นกับน้ำหนัก
        port_daily_ret = (out_sample @ w).copy()
        port_daily_ret.iloc[0] -= cost  # หักค่าธรรมเนียมวันแรก
        daily_portfolio_returns[s].append(port_daily_ret)
        
    daily_portfolio_returns["Benchmark"].append(bench_out)

# รวมเป็น DataFrame ผลตอบแทนรายวันตลอด 7 ปี
backtest_returns = pd.DataFrame({
    s: pd.concat(daily_portfolio_returns[s]) for s in strategies + ["Benchmark"]
})

print("Backtest เสร็จสมบูรณ์!")
print(f"ช่วงเวลา: {backtest_returns.index[0].strftime('%Y-%m-%d')} ถึง {backtest_returns.index[-1].strftime('%Y-%m-%d')}")
print(f"จำนวนวันทำการที่ทดสอบจริง: {len(backtest_returns)} วัน")

### Step 3: การประเมินผลการดำเนินงานและตัวชี้วัดความเสี่ยง (Performance & Risk Metrics)

#### อธิบายตัวชี้วัดในตารางสรุปผล:
1. **Annualized Return (%)**: ผลตอบแทนเฉลี่ยต่อปี (คำนวณจาก $\bar{R}_{daily} \times 252$)
2. **Annualized Volatility (%)**: ความผันผวนต่อปี (คำนวณจาก $\sigma_{daily} \times \sqrt{252}$)
3. **Sharpe Ratio (Rf=2.5%)**: อัตราส่วนผลตอบแทนส่วนเพิ่มเทียบความเสี่ยง โดยใช้ Risk-Free Rate ของไทย $2.5\%$:
   $$\text{Sharpe} = \frac{R_p - 0.025}{\sigma_p}$$
4. **Max Drawdown (%)**: จุดขาดทุนลึกที่สุดที่พอร์ตเคยตกลงมาจากยอดดอยสูงสุดในอดีต (Peak-to-Trough)
5. **Calmar Ratio**: อัตราส่วนความคุ้มค่าระหว่างผลตอบแทนต่อปีกับ Max Drawdown ($\frac{\text{Ann. Return}}{|\text{Max Drawdown}|}$) ยิ่งสูงแปลว่าฟื้นตัวเร็วและทนทาน
6. **Annual Turnover (%)**: ปริมาณการซื้อขายเฉลี่ยต่อปี (Turnover เฉลี่ยต่อเดือน $\times 12$) บ่งบอกถึงต้นทุนแฝงในการบริหารกองทุน

In [ ]:
# ตารางสรุปผลการดำเนินงานตลอด 7 ปี (Net of Costs)
ann_ret = calculate_annualized_return(backtest_returns) * 100
ann_vol = calculate_annualized_volatility(backtest_returns) * 100
sharpe = calculate_sharpe_ratio(backtest_returns, risk_free_rate=0.025)
mdd = calculate_max_drawdown(backtest_returns) * 100
calmar = ann_ret / mdd.abs()

# คำนวณ Turnover เฉลี่ยต่อปี
ann_turnover = pd.Series({s: np.mean(monthly_turnovers[s]) * 12 * 100 for s in strategies})
ann_turnover["Benchmark"] = 0.0

summary_df = pd.DataFrame({
    "Ann. Return (%)": ann_ret,
    "Ann. Volatility (%)": ann_vol,
    "Sharpe Ratio (Rf=2.5%)": sharpe,
    "Max Drawdown (%)": mdd,
    "Calmar Ratio": calmar,
    "Ann. Turnover (%)": ann_turnover
}).round(2)

print("=== ตารางสรุปผล 7-Year Walk-Forward Backtest (2018–2024, Net of Costs) ===")
summary_df

### Step 4: การพล็อต Equity Curves และ Underwater Drawdowns

#### อธิบายโค้ดกราฟในเซลล์ถัดไป:
1. **`cum_returns = (1 + backtest_returns).cumprod()`**:
   - คำนวณ **Wealth Index**: จำลองว่าถ้าเริ่มต้นลงทุนด้วยเงิน 1 บาท ณ วันที่ 4 มกราคม 2018 เมื่อเวลาผ่านไปแต่ละวัน เงินลงทุนของเราจะเติบโตสะสมกลายเป็นเท่าไหร่
2. **`drawdowns = calculate_drawdowns(backtest_returns) * 100`**:
   - คำนวณกราฟใต้น้ำ (Underwater Curve) วัดระยะห่างจากจุดสูงสุดเดิมว่า ณ ขณะนั้นพอร์ตกำลังติดลบอยู่กี่เปอร์เซ็นต์
3. **`plt.subplots(2, 1, ...)`**:
   - กราฟบน: เส้นการเติบโตของเงิน 1 บาทในแต่ละกลยุทธ์
   - กราฟล่าง: กราฟ Drawdown เพื่อดูว่าช่วงวิกฤต เช่น COVID เดือนมีนาคม 2020 แต่ละพอร์ตตกลงไปลึกแค่ไหน

In [ ]:
# พล็อต Equity Curves & Drawdown Comparison
cum_returns = (1 + backtest_returns).cumprod()
drawdowns = calculate_drawdowns(backtest_returns) * 100

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True, gridspec_kw={"height_ratios": [2, 1]})

# 1. Equity Curves
for col in cum_returns.columns:
    lw = 2.5 if col in ["MaxSharpe", "Benchmark"] else 1.5
    ax1.plot(cum_returns.index, cum_returns[col], label=col, linewidth=lw)
ax1.set_title("7-Year Walk-Forward Cumulative Growth of 1 THB (2018–2024, Net of Costs)", fontsize=14, fontweight="bold")
ax1.set_ylabel("Portfolio Value (THB)")
ax1.grid(True, alpha=0.3)
ax1.legend(loc="upper left")

# 2. Underwater Drawdown
for col in drawdowns.columns:
    ax2.plot(drawdowns.index, drawdowns[col], label=col, alpha=0.8)
ax2.set_title("Historical Underwater Drawdown (%)", fontsize=12)
ax2.set_ylabel("Drawdown (%)")
ax2.set_xlabel("Date")
ax2.grid(True, alpha=0.3)
ax2.legend(loc="lower left")

plt.tight_layout()
plt.show()

### Notes: บทวิเคราะห์ผลลัพธ์สำคัญ (Quant Performance Insights & Interview Takeaways)

1. **ภาวะตลาดหุ้นไทยตลอด 7 ปี (2018–2024)**:
   - **Benchmark (`TDEX.BK`)** โตเพียง **+1.49% ต่อปี** ซึ่งต่ำกว่าอัตราผลตอบแทนปราศจากความเสี่ยง (Risk-free Rate 2.5%)
   - ส่งผลให้ **Sharpe Ratio ของตลาดติดลบ (-0.07)** และมี Max Drawdown ลึกถึง **-36.21%**

2. **ทุกกลยุทธ์จัดพอร์ตเชิงปริมาณสามารถเอาชนะ Benchmark ได้ (Alpha Generation)**:
   - **Equal-Weight (EW)**: ได้ผลตอบแทน **+5.44% ต่อปี** (ชนะตลาดเกือบ 4 เท่า) ด้วย **Turnover ต่ำมากเพียง 7.14% ต่อปี** เหมาะกับกลยุทธ์ต้นทุนต่ำแบบ Passive
   - **Minimum-Variance (Min-Var)**: ลดความผันผวนลงเหลือ **15.25% ต่อปี** (ต่ำที่สุดในบรรดาพอร์ตหุ้นทั้งหมด ต่ำกว่า Benchmark) ตอกย้ำพลังของการกระจายความเสี่ยงข้ามอุตสาหกรรม
   - **Maximum-Sharpe (Max-Sharpe)**: โดดเด่นที่สุดในแง่ผลตอบแทน **+10.75% ต่อปี** และ **Sharpe Ratio สูงถึง 0.42** (สร้างผลตอบแทนทบต้น 7 ปีได้ถึง **+79.4%**) ถึงแม้จะมี Turnover สูง (126.95% ต่อปี) แต่ Excess Return ก็สูงพอที่จะชดเชยค่าคอมมิชชั่นได้อย่างสบาย